# Yu-Gi-Oh Rarity Multitask Baseline (11 Binary Models)

Notebook này triển khai baseline end-to-end theo đúng ý tưởng:
- Feature engineering từ `train.csv` để tạo 8 binary sub-target cho Task 1 (rarity).
- Tạo 3 dataset crop khác nhau: **name**, **art**, **full**.
- Thiết kế 3 augmentation pipeline khác nhau theo từng vùng crop.
- Train **11 model binary** (8 sub-target rarity + 3 task foil).
- Ensemble `resnet18` + `efficientnet_b4`.
- Inference + hậu xử lý để suy ra rarity từ 8 sub-target và consistency check với task 2/3/4.

> Lưu ý: notebook giả định structure dữ liệu:
> - `data/train/images/*.jpg`
> - `data/test/images/*.jpg`
> - `data/train.csv`
> - `data/sample_submission.csv`

In [ ]:
# Nếu chạy trên Colab/Kaggle, bật cell này khi cần
# !pip install -q timm albumentations opencv-python-headless scikit-learn tqdm

In [ ]:

import os
import random
from dataclasses import dataclass
from typing import Dict

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


In [ ]:

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


In [ ]:

TRAIN_CSV = 'data/train.csv'
TRAIN_IMG_DIR = 'data/train/images'
TEST_IMG_DIR = 'data/test/images'
SAMPLE_SUB = 'data/sample_submission.csv'
OUTPUT_DIR = 'outputs_baseline'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RARITY_TO_ID = {
    'Common': 1,
    'Rare': 2,
    'Super Rare': 3,
    'Ultra Rare': 4,
    'Secret Rare': 5,
    'Prismatic Secret Rare': 6,
    'Quarter Century Secret Rare': 7,
    'Starlight Rare': 8,
}


## 1) Feature engineering từ rarity -> 8 binary sub-target + 3 foil targets

In [ ]:

RARITY_META = {
    1: dict(name_foil=0, art_foil=0, full_foil=0, name_silver=0, name_rainbow=0, art_shiny=0, art_diag=0, art_grid=0, art_fullcard_hline=0, full_25th=0),
    2: dict(name_foil=1, art_foil=0, full_foil=0, name_silver=1, name_rainbow=0, art_shiny=0, art_diag=0, art_grid=0, art_fullcard_hline=0, full_25th=0),
    3: dict(name_foil=0, art_foil=1, full_foil=0, name_silver=0, name_rainbow=0, art_shiny=1, art_diag=0, art_grid=0, art_fullcard_hline=0, full_25th=0),
    4: dict(name_foil=1, art_foil=1, full_foil=0, name_silver=1, name_rainbow=0, art_shiny=1, art_diag=0, art_grid=0, art_fullcard_hline=0, full_25th=0),
    5: dict(name_foil=1, art_foil=1, full_foil=0, name_silver=0, name_rainbow=1, art_shiny=1, art_diag=1, art_grid=0, art_fullcard_hline=0, full_25th=0),
    6: dict(name_foil=1, art_foil=1, full_foil=0, name_silver=0, name_rainbow=1, art_shiny=1, art_diag=0, art_grid=1, art_fullcard_hline=0, full_25th=0),
    7: dict(name_foil=1, art_foil=1, full_foil=1, name_silver=0, name_rainbow=1, art_shiny=1, art_diag=0, art_grid=0, art_fullcard_hline=1, full_25th=1),
    8: dict(name_foil=1, art_foil=1, full_foil=1, name_silver=0, name_rainbow=1, art_shiny=1, art_diag=0, art_grid=0, art_fullcard_hline=1, full_25th=0),
}

SUBTASKS = ['name_silver','name_rainbow','art_shiny','art_diag','art_grid','art_fullcard_hline','full_foil','full_25th']
FOIL_TASKS = ['name_foil', 'art_foil', 'full_foil']
ALL_TASKS = SUBTASKS + FOIL_TASKS

def attach_targets(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ALL_TASKS:
        out[col] = out['rarity'].map(lambda x: RARITY_META[int(x)][col])
    return out


## 2) Crop 3 dataset: name / art / full

In [ ]:

CROP_CFG = {
    'name': dict(x1=0.06, y1=0.04, x2=0.94, y2=0.13),
    'art':  dict(x1=0.10, y1=0.17, x2=0.90, y2=0.56),
    'full': dict(x1=0.00, y1=0.00, x2=1.00, y2=1.00),
}

def crop_by_region(img: np.ndarray, region: str) -> np.ndarray:
    h, w = img.shape[:2]
    cfg = CROP_CFG[region]
    x1, y1 = int(cfg['x1'] * w), int(cfg['y1'] * h)
    x2, y2 = int(cfg['x2'] * w), int(cfg['y2'] * h)
    return img[y1:y2, x1:x2]


## 3) 3 augmentation khác nhau cho name / art / full

In [ ]:

IMG_SIZE = 384

def get_transforms(region='full', train=True):
    if region == 'name':
        aug_train = A.Compose([
            A.Resize(128, IMG_SIZE),
            A.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.25, hue=0.03, p=0.8),
            A.RandomGamma(gamma_limit=(80, 120), p=0.5),
            A.GaussNoise(std_range=(0.01, 0.03), p=0.25),
            A.Normalize(), ToTensorV2(),
        ])
        aug_valid = A.Compose([A.Resize(128, IMG_SIZE), A.Normalize(), ToTensorV2()])
    elif region == 'art':
        aug_train = A.Compose([
            A.Resize(256, 256),
            A.ColorJitter(brightness=0.15, contrast=0.20, saturation=0.20, hue=0.02, p=0.8),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.Sharpen(alpha=(0.1, 0.3), p=0.2),
            A.GaussNoise(std_range=(0.01, 0.03), p=0.2),
            A.Normalize(), ToTensorV2(),
        ])
        aug_valid = A.Compose([A.Resize(256, 256), A.Normalize(), ToTensorV2()])
    else:
        aug_train = A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.ColorJitter(brightness=0.12, contrast=0.15, saturation=0.15, hue=0.02, p=0.7),
            A.RandomGamma(gamma_limit=(85, 115), p=0.4),
            A.GaussNoise(std_range=(0.01, 0.02), p=0.2),
            A.Normalize(), ToTensorV2(),
        ])
        aug_valid = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(), ToTensorV2()])
    return aug_train if train else aug_valid


## 4) Dataset / Model / Train loop (BCE + AdamW + SWA/TTA hooks)

In [ ]:

class CardBinaryDataset(Dataset):
    def __init__(self, df, img_dir, region, target_col=None, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.region = region
        self.target_col = target_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id'])
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = crop_by_region(img, self.region)
        if self.transform is not None:
            img = self.transform(image=img)['image']
        if self.target_col is None:
            return img, row['id']
        return img, torch.tensor(np.float32(row[self.target_col]))

class BinaryBackbone(nn.Module):
    def __init__(self, model_name='resnet18', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=1)
    def forward(self, x):
        return self.backbone(x).squeeze(-1)

class EnsembleBinary(nn.Module):
    def __init__(self, m1='resnet18', m2='tf_efficientnet_b4.ns_jft_in1k', pretrained=True):
        super().__init__()
        self.model1 = BinaryBackbone(m1, pretrained=pretrained)
        self.model2 = BinaryBackbone(m2, pretrained=pretrained)
    def forward(self, x):
        return 0.5 * (self.model1(x) + self.model2(x))

def binary_f1(y_true, y_prob, thr=0.5):
    return f1_score(y_true, (y_prob >= thr).astype(int))


In [ ]:

@dataclass
class TrainConfig:
    n_splits: int = 5
    epochs: int = 8
    batch_size: int = 16
    lr: float = 3e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    use_swa: bool = True
    swa_start: int = 5

CFG = TrainConfig()

def train_one_task(train_df, task_col, region, model_save_dir):
    os.makedirs(model_save_dir, exist_ok=True)
    oof = np.zeros(len(train_df), dtype=np.float32)
    skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=SEED)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['rarity'].values)):
        tr_df = train_df.iloc[tr_idx].copy()
        va_df = train_df.iloc[va_idx].copy()
        dl_tr = DataLoader(CardBinaryDataset(tr_df, TRAIN_IMG_DIR, region, task_col, get_transforms(region, True)),
                           batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=True)
        dl_va = DataLoader(CardBinaryDataset(va_df, TRAIN_IMG_DIR, region, task_col, get_transforms(region, False)),
                           batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

        model = EnsembleBinary(pretrained=True).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
        best_f1, best_path = -1, os.path.join(model_save_dir, f'{task_col}_{region}_fold{fold}.pth')

        for epoch in range(CFG.epochs):
            model.train()
            for x, y in dl_tr:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()

            model.eval()
            probs_list, ys_list = [], []
            with torch.no_grad():
                for x, y in dl_va:
                    x = x.to(DEVICE)
                    probs_list.append(torch.sigmoid(model(x)).cpu().numpy())
                    ys_list.append(y.numpy())
            va_prob = np.concatenate(probs_list)
            va_true = np.concatenate(ys_list)
            va_f1 = binary_f1(va_true, va_prob)
            if va_f1 > best_f1:
                best_f1 = va_f1
                torch.save(model.state_dict(), best_path)
            print(f'[{task_col}][{region}] fold={fold} epoch={epoch} f1={va_f1:.4f}')

        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
        model.eval()
        pred = []
        with torch.no_grad():
            for x, _ in dl_va:
                pred.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
        oof[va_idx] = np.concatenate(pred)

    print(f'OOF F1 {task_col}:', binary_f1(train_df[task_col].values, oof))
    return oof


## 5) Train 11 models

In [ ]:

TASK_REGION = {
    'name_silver': 'name', 'name_rainbow': 'name',
    'art_shiny': 'art', 'art_diag': 'art', 'art_grid': 'art',
    'art_fullcard_hline': 'full',
    'full_foil': 'full', 'full_25th': 'full',
    'name_foil': 'name', 'art_foil': 'art',
}

train_df = pd.read_csv(TRAIN_CSV)
if train_df['rarity'].dtype == object:
    train_df['rarity'] = train_df['rarity'].map(RARITY_TO_ID)
train_df = attach_targets(train_df)

OOF = {}
for task in ALL_TASKS:
    region = TASK_REGION[task]
    OOF[task] = train_one_task(train_df, task, region, os.path.join(OUTPUT_DIR, 'weights', task))


## 6) Suy ra rarity từ 8 sub-target + consistency check với task 2/3/4

In [ ]:

RARITY_PATTERN = {tuple(meta[t] for t in SUBTASKS): rid for rid, meta in RARITY_META.items()}

def infer_rarity_from_subtasks(row_bin: Dict[str, int]):
    return RARITY_PATTERN.get(tuple(row_bin[t] for t in SUBTASKS), None)

def foil_triplet_to_unique_rarity(name_foil, art_foil, full_foil):
    cands = [rid for rid, m in RARITY_META.items() if (m['name_foil'], m['art_foil'], m['full_foil']) == (name_foil, art_foil, full_foil)]
    return cands[0] if len(cands) == 1 else None

def postprocess_rarity(pred_bin_subtasks, pred_foil):
    rarity = infer_rarity_from_subtasks(pred_bin_subtasks)
    triplet = (pred_foil['name_foil'], pred_foil['art_foil'], pred_foil['full_foil'])
    if rarity is not None:
        mt = (RARITY_META[rarity]['name_foil'], RARITY_META[rarity]['art_foil'], RARITY_META[rarity]['full_foil'])
        if mt == triplet:
            return rarity
    unique_r = foil_triplet_to_unique_rarity(*triplet)
    if unique_r is not None:
        return unique_r
    best_r, best_d = None, 999
    for rid, meta in RARITY_META.items():
        d = sum(int(pred_bin_subtasks[t] != meta[t]) for t in SUBTASKS)
        if d < best_d:
            best_d, best_r = d, rid
    return best_r


## 7) Inference + TTA + Submission

In [ ]:

def load_fold_models(task, region):
    models = []
    for fold in range(CFG.n_splits):
        p = os.path.join(OUTPUT_DIR, 'weights', task, f'{task}_{region}_fold{fold}.pth')
        m = EnsembleBinary(pretrained=False).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models.append(m)
    return models

def tta_predict(models, x):
    with torch.no_grad():
        p1 = np.mean([torch.sigmoid(m(x)).cpu().numpy() for m in models], axis=0)
        x_flip = torch.flip(x, dims=[3])
        p2 = np.mean([torch.sigmoid(m(x_flip)).cpu().numpy() for m in models], axis=0)
    return 0.5 * (p1 + p2)

def predict_task(test_df, task, region):
    ds = CardBinaryDataset(test_df, TEST_IMG_DIR, region, None, get_transforms(region, False))
    dl = DataLoader(ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
    models = load_fold_models(task, region)
    ids, probs = [], []
    for x, batch_ids in tqdm(dl, desc=f'predict {task}'):
        x = x.to(DEVICE)
        p = tta_predict(models, x)
        probs.extend(p.tolist())
        ids.extend(batch_ids)
    return pd.DataFrame({'id': ids, f'{task}_prob': probs})

sample = pd.read_csv(SAMPLE_SUB)
test_df = sample[['id']].copy()
pred_df = test_df.copy()
for task in ALL_TASKS:
    p = predict_task(test_df, task, TASK_REGION[task])
    pred_df = pred_df.merge(p, on='id', how='left')

for task in ALL_TASKS:
    pred_df[task] = (pred_df[f'{task}_prob'] >= 0.5).astype(int)

rows = []
for _, r in pred_df.iterrows():
    sub_bin = {t: int(r[t]) for t in SUBTASKS}
    foil_bin = {t: int(r[t]) for t in FOIL_TASKS}
    rarity = postprocess_rarity(sub_bin, foil_bin)
    rows.append([r['id'], rarity, foil_bin['name_foil'], foil_bin['art_foil'], foil_bin['full_foil']])

sub = pd.DataFrame(rows, columns=['id', 'rarity', 'name_foil', 'art_foil', 'full_foil'])
sub.to_csv(os.path.join(OUTPUT_DIR, 'submission.csv'), index=False)
sub.head()


## 8) Notes

- Có thể tune threshold từng task theo OOF.
- Có thể thêm class weight cho BCE khi class lệch.
- Có thể tinh chỉnh ROI để bắt dấu `25th` tốt hơn.